# Preprocessing pipelines

Transforms are the preprocessing layer: composable, non-mutating operations on a single scene. Each one operates on a **single-scene** `dict` (one sample, before collation), is single-purpose, and never mutates its input. You chain them with `Compose`, and every non-trivial transform has a tensor-level twin under `torch_pointcloud.transforms.functional`.

This notebook builds intuition step by step on a synthetic scene, so every cell runs on CPU. The [Transforms gallery](../transforms/overview.md) shows before/after pictures of the full catalog.

In [ ]:
import torch

import torch_pointcloud.transforms as T

torch.manual_seed(0)

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## A scene is a dict

We use the standard keys: `pos` for $(N, 3)$ coordinates and `color` for $(N, 3)$ RGB. Here is a synthetic "wall + floor" scene with a color gradient, so transforms are easy to see.

In [ ]:
n = 8000
floor = torch.rand(n // 2, 3) * torch.tensor([4.0, 4.0, 0.05])
wall = torch.rand(n // 2, 3) * torch.tensor([4.0, 0.05, 2.5])
pos = torch.cat([floor, wall])
color = (pos - pos.min(0).values) / (pos.max(0).values - pos.min(0).values)  # position -> RGB

scene = {"pos": pos, "color": color}
{k: tuple(v.shape) for k, v in scene.items()}

In [ ]:
show_cloud(scene["pos"], color=scene["color"], title="input scene", size=2);

`show_cloud` reads an $(N, 3)$ `color` as per-point RGB: the wall stands at $y \approx 0$ and the floor runs away from it, and the gradient makes both easy to follow through a transform.

## One transform at a time

A transform is constructed with the `keys` it acts on, then called on the dict. It returns a **new** dict; keys it does not touch pass through untouched. `Rescale(method="centroid")` centers the cloud and scales it into the unit sphere:

In [ ]:
rescaled = T.Rescale(keys="pos", method="centroid")(scene)

print("input  center / radius:", scene["pos"].mean(0).round(decimals=2).tolist(), "/", round(scene["pos"].norm(dim=1).max().item(), 2))
print("output center / radius:", rescaled["pos"].mean(0).round(decimals=2).tolist(), "/", round(rescaled["pos"].norm(dim=1).max().item(), 2))
print("color untouched:", torch.equal(rescaled["color"], scene["color"]))

![A room point cloud before and after Rescale(method="centroid"), the second one centered and much smaller.](../assets/transforms/rescale_centroid_scene.png)

`Rescale` moves the centroid to the origin and divides by the largest distance from it, so the output always sits in the unit sphere: on this scene the center goes from $(2.03, 1.00, 0.64)$ to the origin and the radius from 5.64 to 1.0. That is what puts scenes of different sizes on one scale. Nothing about the shape changes, which is why the picture above needs both panels drawn in the same box to show anything.

When several keys are passed together, they stay in correspondence. `RandomSample` draws the same indices for every listed key, so `pos` and `color` shrink to the same 2048 rows:

In [ ]:
sampled = T.RandomSample(keys=("pos", "color"), num_samples=2048)(rescaled)
{k: tuple(v.shape) for k, v in sampled.items()}

## Augmentations

Augmentations are random and take a probability `p`. With `p=1.0` they always fire (handy for a demo). Below: a 30 degrees rotation about the vertical axis, then a color jitter. Color transforms act on the `color` key and expect $[0, 1]$ floats by default (pass `int_color=True` for $[0, 255]$).

Each step below is the input of the next, so one figure shows the whole chain:

In [ ]:
rotated = T.RandomRotate(keys="pos", angle_range=(30.0, 30.0), axis=2, p=1.0)(sampled)
jittered = T.RandomColorJitter(keys="color", brightness=0.5, contrast=0.5, saturation=0.5, p=1.0)(rotated)

steps = {
    "input": scene,
    "+ RandomSample": sampled,
    "+ RandomRotate": rotated,
    "+ RandomColorJitter": jittered,
}
fig = plt.figure(figsize=(12, 3.2))
for i, (title, step) in enumerate(steps.items()):
    show_cloud(step["pos"], color=step["color"], ax=fig.add_subplot(1, 4, i + 1, projection="3d"), title=title, size=2)

![The synthetic scene through four steps of one pipeline: input, random sample, random rotate and color jitter.](../assets/tutorials/transforms_pipeline_steps.png)

Read the panels left to right. `RandomSample` thins the cloud from 8,000 points to 2,048 without moving any of them, `RandomRotate` turns the whole scene 30 degrees about $z$, and only the last step touches `color`: the ones before it move rows or drop them and leave the RGB values exactly as they were.

`Rescale` runs between the first and the second panel and gets none of its own. Each panel is framed to its own data, which draws every cloud at the same size on screen, and a uniform scaling is then invisible: the numbers above are the whole of it, a center at $(2.03, 1.00, 0.64)$ and a radius of 5.64 going to the origin and 1.0.

## Compose a pipeline

`Compose` chains transforms into one callable. A typical training pipeline normalizes geometry, subsamples to a fixed budget, then augments:

In [ ]:
train_pipeline = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSample(keys=("pos", "color"), num_samples=2048),
    T.RandomFlip(keys="pos", axes=[0, 1], p=0.5),
    T.RandomScale(keys="pos", scale_range=(0.9, 1.1)),
    T.RandomJitter(keys="pos", sigma=0.01, clip=0.05),
])

out = train_pipeline({"pos": pos.clone(), "color": color.clone()})
{k: tuple(v.shape) for k, v in out.items()}

At evaluation you keep the deterministic steps and drop the random ones, so results are reproducible:

In [ ]:
eval_pipeline = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSample(keys=("pos", "color"), num_samples=2048),
])
tuple(eval_pipeline({"pos": pos.clone(), "color": color.clone()})["pos"].shape)

The evaluation pipeline returns the same scene every time; the training one returns a different draw on every call. Run the two side by side, three consecutive draws against one evaluation pass:

In [ ]:
torch.manual_seed(1)
draws = {"eval pipeline": eval_pipeline({"pos": pos.clone(), "color": color.clone()})}
for i in (1, 2, 3):
    draws[f"train pipeline, draw {i}"] = train_pipeline({"pos": pos.clone(), "color": color.clone()})

for name, draw in draws.items():
    print(f"{name}: radius {draw['pos'].norm(dim=1).max():.2f}")

fig = plt.figure(figsize=(12, 3.2))
for i, (title, draw) in enumerate(draws.items()):
    show_cloud(draw["pos"], color=draw["color"], ax=fig.add_subplot(1, 4, i + 1, projection="3d"), title=title, size=2)

![One panel of the evaluation pipeline output beside three draws of the training pipeline, each mirrored a different way.](../assets/tutorials/transforms_train_vs_eval.png)

The evaluation panel is the scene as it stands, subsampled and nothing more, and it comes back identical on every call. Follow the color gradient across the other three: `RandomFlip` fires on all of them, mirroring about $x$ in the first and third draw and about $y$ in the second. `RandomScale` is the change the picture does not show, because each panel is framed to its own data; the printed radii are where it shows up, 0.98, 0.89 and 0.91 against the 0.99 of the evaluation pass. That variety is the whole point: the model never sees the same scene twice.

## Reproducibility

Random transforms accept a `torch.Generator`, so a pipeline can be made deterministic without touching the global RNG. Two draws from the same seed match:

In [ ]:
g1 = torch.Generator().manual_seed(42)
g2 = torch.Generator().manual_seed(42)
a = T.RandomSample(keys="pos", num_samples=512, generator=g1)({"pos": pos.clone()})["pos"]
b = T.RandomSample(keys="pos", num_samples=512, generator=g2)({"pos": pos.clone()})["pos"]
print("identical draws:", torch.equal(a, b))

## The functional layer

If you already hold a tensor and do not want a dict, call the functions in `torch_pointcloud.transforms.functional` directly. They are the same operations the class transforms wrap.

In [ ]:
import torch_pointcloud.transforms.functional as F

p = torch.randn(1000, 3)
p = F.shift(p, method="bbox", axes=[0, 1])  # center X and Y on the bbox midpoint
p = F.shift(p, method="min", axes=[2])       # drop Z so the floor sits at 0
mask = F.sphere_mask(p, center=[0.0, 0.0, 0.0], radius=2.0)
print("shifted:", tuple(p.shape), "| kept by sphere mask:", int(mask.sum()))

## Next steps

- Browse every transform with before/after pictures in the [Transforms gallery](../transforms/overview.md).
- Each pretrained checkpoint records its own pipeline, returned as the `transform` entry of `create_model(..., return_info=True)`.
- Plug a pipeline into a dataset in [Use your own data](04-custom-dataset.md), then train with it in [Train a model](05-training.md).